<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!pip install -q duckdb
import duckdb, pandas as pd, numpy as np, os
from google.colab import userdata
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
month = "2026-03"
daily_m = f"read_parquet('{rel}/fact_content_daily_performance/month={month}/data_0.parquet')"
content = f"read_parquet('{rel}/dim_content.parquet')"
os.makedirs("work/outputs", exist_ok=True)
df = con.sql(f"""
SELECT d.content_hash_id, d.client_hash_id,
       SUM(d.gsc_clicks) AS clicks, SUM(d.gsc_impressions) AS impressions,
       SUM(d.gsc_sum_position) / NULLIF(SUM(d.gsc_impressions), 0) AS avg_position,
       COUNT(DISTINCT d.report_date) AS days_seen,
       ANY_VALUE(c.word_count) AS word_count, ANY_VALUE(c.search_volume) AS search_volume,
       ANY_VALUE(c.competition) AS competition, ANY_VALUE(c.backlinks) AS backlinks,
       ANY_VALUE(c.content_type) AS content_type, ANY_VALUE(c.main_intent) AS main_intent,
       ANY_VALUE(c.content_updated_date) AS updated_date, ANY_VALUE(c.is_published) AS is_published,
       ANY_VALUE(c.is_deleted) AS is_deleted
FROM {daily_m} d
LEFT JOIN {content} c ON d.content_hash_id = c.content_hash_id AND d.client_hash_id = c.client_hash_id
WHERE d.gsc_data_available IS TRUE
GROUP BY 1, 2
HAVING SUM(d.gsc_impressions) >= 100
""").df()
df["ctr"] = df["clicks"] / df["impressions"] * 100
print(f"{len(df):,} pages in scope, {df['client_hash_id'].nunique()} clients")
print(df[["impressions", "clicks", "ctr", "avg_position", "word_count"]].describe().round(2).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 pages in scope, 44 clients
       impressions     clicks        ctr  avg_position  word_count
count    101441.00  101441.00  101441.00     101441.00     72449.0
mean       2748.36       8.04       0.26         14.35     2966.15
std        6945.16      34.89       0.42         14.80     1093.12
min         100.00       0.00       0.00          0.02         0.0
25%         283.00       0.00       0.00          4.77      2479.0
50%         786.00       1.00       0.12          8.20      2773.0
75%        2514.00       6.00       0.37         18.95      3224.0
max      617124.00    5668.00      15.58        106.89     29341.0


## 1. My rule and its reason codes

Two signal checks before I write any rule.

Signal 1: CTR falls as position worsens. Verdict: CONFIRMED. Median CTR drops cleanly across position buckets, 0.211 at positions 1-3 down to 0.000 beyond 21. Zero-click share rises the other way, 22.9% to 88.8%. This is the signal behind FlyRank's CTR-fix logic and it holds in my slice. It also means raw CTR is useless as a ranking signal on its own, since it would just re-rank pages by where they already sit.

Signal 2: stale pages underperform. Verdict: FALSE, and not for the reason I expected. I cannot test this. Only 18,111 of 101,441 pages carry a usable content_updated_date after the join, and of those, 17,969 sit under 90 days. The 90-180d bucket has 124 pages and the 180-365d bucket has 18. That is not a staleness distribution, it is one bucket and noise.

Reading the 0.082 median CTR of those 18 pages as "older pages do fine" would be exactly the mistake this check exists to prevent. Eighteen rows cannot support any verdict.

What this changes. Staleness is out of my rule entirely. Any refresh-style logic that leans on last-updated dates cannot be built on this slice, and I would rather say so than compute a number from 18 rows.

In [10]:
df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df["updated_date"])).dt.days
print("SIGNAL 1: CTR vs position")
df["pos_bucket"] = pd.cut(df["avg_position"], [0, 3, 10, 20, 50, 200], labels=["1-3", "4-10", "11-20", "21-50", "50+"])
print(df.groupby("pos_bucket", observed=True).agg(n=("ctr", "size"), median_ctr=("ctr", "median"), zero_click_share=("clicks", lambda s: (s == 0).mean())).round(3).to_string())
print()
print("SIGNAL 2: staleness vs CTR")
df["stale_bucket"] = pd.cut(df["days_since_update"], [-1, 90, 180, 365, 730, 99999], labels=["<90d", "90-180d", "180-365d", "1-2y", "2y+"])
print(df.groupby("stale_bucket", observed=True).agg(n=("ctr", "size"), median_ctr=("ctr", "median"), median_pos=("avg_position", "median")).round(3).to_string())
print(f"\ndays_since_update missing: {df['days_since_update'].isna().sum():,} ({df['days_since_update'].isna().mean():.1%})")

SIGNAL 1: CTR vs position
                n  median_ctr  zero_click_share
pos_bucket                                     
1-3         10194       0.211             0.229
4-10        47811       0.190             0.279
11-20       19547       0.101             0.428
21-50       19758       0.000             0.511
50+          4131       0.000             0.888

SIGNAL 2: staleness vs CTR
                  n  median_ctr  median_pos
stale_bucket                               
<90d          17969       0.061       8.492
90-180d         124       0.000      19.980
180-365d         18       0.082       6.717

days_since_update missing: 0 (0.0%)


## 2. Build the ranked queue (writes the CSV)

The rule in plain words. A page is worth reviewing if it is visible enough to matter, it sits high enough that a click was realistically available, and it converts fewer of those impressions than other pages at a similar position.

Three conditions, no fitted weights. Score is the shortfall against the median CTR of the page's own position bucket, multiplied by impressions, so a small shortfall on a high-traffic page can outrank a large shortfall on a small one.

Reason codes: high_volume_low_ctr, top_position_no_clicks, mid_position_shortfall, below_peer_median.

What the score deliberately excludes. No last_optimized_date or optimization_eligible_date, since those encode a decision someone already made. No GA4 or session fields, since those are post-click. Nothing from April onwards; the slice ends 2026-03-31.

In [11]:
work = df[(df["is_deleted"] != True) & (df["avg_position"] <= 50)].copy()
work["pos_bucket"] = pd.cut(work["avg_position"], [0, 3, 10, 20, 50], labels=["1-3", "4-10", "11-20", "21-50"])
peer = work.groupby("pos_bucket", observed=True)["ctr"].median().rename("peer_ctr")
work = work.join(peer, on="pos_bucket")
work["shortfall"] = (work["peer_ctr"] - work["ctr"]).clip(lower=0)
work["score"] = work["shortfall"] * work["impressions"]
def reason(r):
    if r["avg_position"] <= 3 and r["clicks"] == 0:
        return "top_position_no_clicks"
    if r["impressions"] >= 2514 and r["ctr"] < r["peer_ctr"]:
        return "high_volume_low_ctr"
    if r["avg_position"] <= 20 and r["shortfall"] > 0:
        return "mid_position_shortfall"
    return "below_peer_median"
work["reason_code"] = work.apply(reason, axis=1)
work["action"] = "review title and meta description"
q = work[work["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1
cols = ["rank", "content_hash_id", "client_hash_id", "score", "reason_code", "action", "impressions", "clicks", "ctr", "peer_ctr", "shortfall", "avg_position"]
q[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"scored {len(work):,} pages, {len(q):,} have score > 0")
print(f"\nreason code spread:\n{q['reason_code'].value_counts().to_string()}")
print(f"\nwrote work/outputs/baseline_action_score.csv, {len(q):,} rows")
print(f"\npeer medians:\n{peer.round(3).to_string()}")


scored 97,281 pages, 38,745 have score > 0

reason code spread:
reason_code
mid_position_shortfall    28732
high_volume_low_ctr        7685
top_position_no_clicks     2328

wrote work/outputs/baseline_action_score.csv, 38,745 rows

peer medians:
pos_bucket
1-3      0.211
4-10     0.190
11-20    0.101
21-50    0.000


## 3. Top-20 review

How I measured it. The label is April 2026 CTR below the March peer median. The score is built entirely from March. Score on past, label on future. My first attempt labelled pages as underperforming if their March CTR was below peer, which returned precision of exactly 1.000 at every K. That is the leak signature: the label was the score's own two components rebuilt as a boolean. Rebuilt against April and the numbers became real.

Precision@50 is 0.920 against a base rate of 0.445. Roughly twice random. 85,664 of 97,281 scored pages have an April outcome to check against.

The top 20, read with a sceptic's eye. All twenty carry high_volume_low_ctr, which is the volume term doing the work: impressions in the top 20 run from 58,278 to 212,404 against a median of 786 across the slice. Every one sits at position 10 or better, so a click was realistically available.

Rank 11 is the pick I trust least. 289 clicks at a CTR of 0.142 against a peer median of 0.211, so the shortfall is 0.069, the smallest in the top 20. It ranks eleventh purely on 203,497 impressions. A reviewer opening it would find a page already performing at two thirds of its peer group, which is not obviously broken.

Ranks 2, 3 and 6 are the opposite case, and the ones I would send a reviewer to first. Positions 2.693, 0.308 and 0.116, over 80,000 impressions each, and exactly one click apiece. A page at position 0.3 with one click in a month is either mis-titled or serving a query nobody wants answered.

What would make each of these wrong. The same thing in every case: the peer median assumes pages in a position bucket are comparable, and they are not. A branded navigational query at position 1 should convert far better than an informational one, and I have no query intent in this score. Any page whose low CTR is explained by its query type rather than its title is a false positive, and I cannot currently tell which those are.

In [12]:
top = q.head(20)
print(top[["rank", "score", "reason_code", "impressions", "clicks", "ctr", "peer_ctr", "avg_position"]].round(3).to_string(index=False))
print()
lab = work.copy()
apr = con.sql(f"""
SELECT content_hash_id, client_hash_id,
       SUM(gsc_clicks) AS clicks_apr, SUM(gsc_impressions) AS impressions_apr
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet')
WHERE gsc_data_available IS TRUE GROUP BY 1,2 HAVING SUM(gsc_impressions) >= 100
""").df()
apr["ctr_apr"] = apr["clicks_apr"] / apr["impressions_apr"] * 100
lab = lab.merge(apr[["content_hash_id", "client_hash_id", "ctr_apr"]], on=["content_hash_id", "client_hash_id"], how="inner")
lab["label"] = (lab["ctr_apr"] < lab["peer_ctr"]).astype(int)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()
print(f"pages with April outcome: {len(lab):,}")
for k in (10, 20, 50, 100):
    print(f"precision@{k} = {precision_at_k(lab['score'], lab['label'], k):.3f}")
print(f"base rate = {lab['label'].mean():.3f}")
print(f"\nclients in top 50: {q.head(50)['client_hash_id'].nunique()}")
print(f"most common client in top 50 appears {q.head(50)['client_hash_id'].value_counts().iloc[0]} times")

 rank     score         reason_code  impressions  clicks   ctr  peer_ctr  avg_position
    1 42505.708 high_volume_low_ctr     212404.0    24.0 0.011     0.211         0.666
    2 28437.844 high_volume_low_ctr     134984.0     1.0 0.001     0.211         2.693
    3 26131.501 high_volume_low_ctr     124075.0     1.0 0.001     0.211         0.308
    4 22889.924 high_volume_low_ctr     143019.0    43.0 0.030     0.190         3.166
    5 18953.232 high_volume_low_ctr     107584.0    15.0 0.014     0.190         9.736
    6 17623.890 high_volume_low_ctr      83834.0     1.0 0.001     0.211         0.116
    7 16907.795 high_volume_low_ctr     132593.0    83.0 0.063     0.190         5.948
    8 16583.270 high_volume_low_ctr      89332.0     4.0 0.004     0.190         7.832
    9 15329.278 high_volume_low_ctr      83788.0     6.0 0.007     0.190         7.208
   10 14560.837 high_volume_low_ctr      82376.0    11.0 0.013     0.190         8.005
   11 14122.622 high_volume_low_ctr     203

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages with April outcome: 85,664
precision@10 = 0.900
precision@20 = 0.950
precision@50 = 0.920
precision@100 = 0.920
base rate = 0.445

clients in top 50: 11
most common client in top 50 appears 14 times


## 4. Weak picks + leakage check

Weak picks. Rank 11 is the weakest in the top 20: a shortfall of 0.069 against a peer median of 0.211, ranked eleventh on volume alone with 203,497 impressions and 289 clicks. Rank 7 and rank 18 are next at 0.128 and 0.152. The score multiplies shortfall by impressions, so a page already performing at two thirds of its peer group can outrank a genuinely broken one purely on traffic. That is a deliberate choice, since fixing a big page moves more clicks, but it means the queue is not ordered by how wrong a page is, and a reviewer opening rank 11 would find little obviously wrong with it.

The 21-50 bucket cannot enter the queue at all. Its peer median CTR is 0.000, so shortfall is always zero and the score is always zero. 19,758 pages are structurally excluded. Defensible, since a page averaging position 30 has no realistic click to lose, but the rule makes that exclusion silently and I would rather name it than let it pass as a filtering detail.

below_peer_median never fires. The three earlier conditions catch every case, so the fallback is dead code. Three working reason codes, not four.

Leakage check. The score touches peer CTR, own CTR, impressions, position bucket and average position, all computed from March 2026 alone. No last_optimized_date or optimization_eligible_date, both of which encode a decision someone already made about the page. No GA4, session, or AI fields, all of which describe behaviour after a click has happened. Every row feeding the score falls between 2026-03-01 and 2026-03-31, and the label comes from April, strictly after.

The one thing I did get wrong was the label itself. My first version labelled a page underperforming if its March CTR sat below the peer median, which returned precision of exactly 1.000 at every K. The label was the score's own components rebuilt as a boolean. A perfect score is the tell, not the achievement, and rebuilding the label against April was the fix.

Client concentration. The top 50 spans 11 clients, with the largest holding 14 of them, 28%. The full queue spans all 43 clients that were scored, so nothing is structurally excluded, but the top of the list is concentrated. Worth watching if the same client keeps surfacing.

In [13]:
score_inputs = ["peer_ctr", "ctr", "impressions", "avg_position", "pos_bucket", "shortfall"]
banned = ["last_optimized_date", "optimization_eligible_date", "provider_used", "model_used"]
ga4_like = [c for c in df.columns if c.startswith(("ga4_", "sessions_", "ai_")) or c == "scroll_events"]
print("columns the score touches:", score_inputs)
print("product decision flags used:", [c for c in banned if c in score_inputs])
print("post-click fields used:", [c for c in ga4_like if c in score_inputs])
print()
print("date span of every row feeding the score:")
print(con.sql(f"SELECT MIN(report_date) AS first_day, MAX(report_date) AS last_day FROM {daily_m} WHERE gsc_data_available IS TRUE").df().to_string(index=False))
print(f"label month: 2026-04, strictly after the score window")
print()
print("weak picks: smallest shortfalls in the top 20")
print(q.head(20).nsmallest(3, "shortfall")[["rank", "shortfall", "ctr", "peer_ctr", "impressions", "clicks"]].round(3).to_string(index=False))
print()
print("client concentration across the queue")
top50 = q.head(50)["client_hash_id"].value_counts()
print(f"top 50 spans {len(top50)} clients, largest holds {top50.iloc[0]} ({top50.iloc[0]/50:.0%})")
print(f"whole queue spans {q['client_hash_id'].nunique()} clients of {work['client_hash_id'].nunique()} scored")


columns the score touches: ['peer_ctr', 'ctr', 'impressions', 'avg_position', 'pos_bucket', 'shortfall']
product decision flags used: []
post-click fields used: []

date span of every row feeding the score:
 first_day   last_day
2026-03-01 2026-03-31
label month: 2026-04, strictly after the score window

weak picks: smallest shortfalls in the top 20
 rank  shortfall   ctr  peer_ctr  impressions  clicks
   11      0.069 0.142     0.211     203497.0   289.0
    7      0.128 0.063     0.190     132593.0    83.0
   18      0.152 0.060     0.211      70398.0    42.0

client concentration across the queue
top 50 spans 11 clients, largest holds 14 (28%)
whole queue spans 43 clients of 43 scored


In [14]:
import json
metrics = {"month": month, "pages_scored": int(len(work)), "queue_rows": int(len(q)),
           "base_rate": round(float(lab["label"].mean()), 4),
           "precision_at": {str(k): round(float(precision_at_k(lab["score"], lab["label"], k)), 4) for k in (10, 20, 50, 100)},
           "label_source": "2026-04 CTR below March peer median", "score_window": "2026-03"}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "month": "2026-03",
  "pages_scored": 97281,
  "queue_rows": 38745,
  "base_rate": 0.4449,
  "precision_at": {
    "10": 0.9,
    "20": 0.95,
    "50": 0.92,
    "100": 0.92
  },
  "label_source": "2026-04 CTR below March peer median",
  "score_window": "2026-03"
}


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.